# Step 8 — Agentic RAG chatbot (LangGraph)

Normal RAG always runs vector search. **Agentic RAG** lets the LLM **choose tools**, read their results, then answer.

```
User question
    -> LLM decides: vector_search / fulltext_search / graph_search (maybe several)
    -> Neo4j returns evidence + URLs
    -> LLM writes answer WITH citations
    -> or refuses: "This is not covered in the current sources."
```

This notebook uses LangGraph's ReAct agent (`create_react_agent`) wired to the three tools from Step 7.

> Code: `signalpulse/agent.py`

## 1. Concepts

**ReAct loop (Reason + Act).** The model alternates between thinking and tool calls until it has enough evidence to answer. LangGraph runs that loop for us.

**Tool choice.** The model picks tools from their descriptions:
- conceptual → `vector_search`
- CVE / docket / exact id → `fulltext_search`
- "related / connected / issued" → `graph_search`

**Citations.** The system prompt requires every factual claim to include a source title + URL from tool output.

**No-evidence guardrail.** If tools return `NO_EVIDENCE` or nothing useful, the agent must say *exactly*: `This is not covered in the current sources.` — no guessing from world knowledge.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signalpulse import graph
from signalpulse.agent import SYSTEM_PROMPT, TOOLS, ask
from signalpulse.llm import available_providers

graph.verify_connectivity()
print("Neo4j:", graph.node_counts())
print("LLM providers:", available_providers())
print("Tools:", [t.name for t in TOOLS])
print("\n--- system prompt (excerpt) ---\n")
print(SYSTEM_PROMPT[:500], "...")

Neo4j: {'Document': 20, 'Chunk': 29, 'Entity': 122}
LLM providers: ['groq', 'gemini']
Tools: ['vector_search', 'fulltext_search', 'graph_search']

--- system prompt (excerpt) ---

You are SignalPulse AI, an assistant for U.S. public-sector regulatory and cybersecurity intelligence.

You have three retrieval tools over a Neo4j knowledge graph:
- vector_search: semantic / conceptual questions
- fulltext_search: exact ids (CVE-..., docket numbers) and keywords
- graph_search: entity relationships and provenance

Rules (non-negotiable):
1. ALWAYS call at least one tool before answering a factual question. Never answer from memory alone.
2. Base every claim ONLY on tool result ...


## 2. Helper: ask a question and show tool calls + answer

`ask(question)` runs the full agent loop and returns an `AgentReply` with the final answer and which tools were used.

In [2]:
def demo(question: str) -> None:
    print("=" * 72)
    print("Q:", question)
    reply = ask(question)
    print("\nTools called:")
    if not reply.tool_calls:
        print("  (none)")
    for tc in reply.tool_calls:
        print(f"  - {tc['name']}({tc['args']})")
    print("\nAnswer:\n")
    print(reply.answer)
    print(f"\n[refused={reply.refused}]")
    print()

## 3. Live demos (practical employee questions)

These replace artificial "graph sightseeing" prompts with questions a security/compliance/delivery person would actually ask.


In [3]:
demo("What is CVE-2026-56291, what product does it affect, and what's the remediation deadline?")


Q: What is CVE-2026-56291, what product does it affect, and what's the remediation deadline?



Tools called:
  - fulltext_search({'domain': '', 'query': 'CVE-2026-56291'})

Answer:

CVE-2026-56291 is a vulnerability in the Balbooa Forms product, specifically an unrestricted upload of file with dangerous type vulnerability. The remediation deadline is 2026-07-13, as per the BOD 26-04 patching guidelines (https://nvd.nist.gov/vuln/detail/CVE-2026-56291).

[refused=False]



## 4. Conceptual NIST question → expects `vector_search`


In [4]:
demo("According to NIST, how should we structure cybersecurity risk management for a federal or state project?")


Q: According to NIST, how should we structure cybersecurity risk management for a federal or state project?


C:\Users\saihaj\Documents\22nd Project\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange



Tools called:
  - vector_search({'domain': 'Tech Standards & Safety', 'query': 'NIST cybersecurity risk management structure for federal or state projects'})

Answer:

According to NIST, cybersecurity risk management for a federal or state project should be structured using the NIST Cybersecurity Framework (CSF) 2.0, which provides guidance to industry, government agencies, and other organizations (https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf). The framework includes controls such as access control policy and procedures (https://csrc.nist.gov/publications/detail/sp/800-53/rev-5/final), account management (https://csrc.nist.gov/publications/detail/sp/800-53/rev-5/final), and risk management plans (https://csrc.nist.gov/projects/risk-management). Additionally, NIST SP 800-53 Release 5.2.0 includes changes in response to Executive Order 14306 (https://csrc.nist.gov/projects/risk-management).

[refused=False]



## 5. BOD / KEV remediation guidance (may use `graph_search` + others)

A real employee question about required actions — not "what is CISA connected to in the graph?"


In [5]:
demo("Are there CISA-required actions or BOD guidance tied to the latest KEV entries?")


Q: Are there CISA-required actions or BOD guidance tied to the latest KEV entries?



Tools called:
  - vector_search({'domain': '', 'query': 'CISA-required actions or BOD guidance tied to the latest KEV entries'})
  - fulltext_search({'domain': '', 'query': 'CISA-required actions or BOD guidance tied to the latest KEV entries'})
  - graph_search({'entity_name': 'KEV entries'})

Answer:

Based on the vector search and fulltext search results, there are CISA-required actions or BOD guidance tied to the latest KEV entries, specifically:
* CVE-2026-56291: Balbooa Forms Unrestricted Upload of File with Dangerous Type Vulnerability (https://nvd.nist.gov/vuln/detail/CVE-2026-56291)
* CVE-2026-48939: iCagenda Unrestricted Upload of File with Dangerous Type Vulnerability (https://nvd.nist.gov/vuln/detail/CVE-2026-48939)
Stakeholders are responsible for evaluating each asset's internet exposure and ensuring adherence to BOD 26-04 patching guidelines, with a due date of 2026-07-13 (https://nvd.nist.gov/vuln/detail/CVE-2026-56291, https://nvd.nist.gov/vuln/detail/CVE-2026-48939).

## 6. No-evidence guardrail (should refuse)


In [6]:
demo("What is the capital of France?")

Q: What is the capital of France?



Tools called:
  - vector_search({'domain': '', 'query': 'capital of France'})

Answer:

This is not covered in the current sources.

[refused=True]



## 7. Full practical eval suite (15 + 2 refusals)

The question list lives in `signalpulse/eval_questions.py`. Run:

```bash
python run_eval.py
```

Results are saved to `data/processed/eval_results.json`. The next cell loads that file and prints a scoreboard.


In [7]:
from pathlib import Path
import json
from collections import Counter

path = ROOT / "data" / "processed" / "eval_results.json"
if not path.exists():
    print("No eval_results.json yet. Run: python run_eval.py")
else:
    rows = json.loads(path.read_text(encoding="utf-8"))
    print("Labels:", dict(Counter(r["label"] for r in rows)))
    print()
    print(f"{'ID':4} {'LABEL':18} {'TOOLS':42} QUESTION")
    print("-" * 110)
    for r in rows:
        tools = ", ".join(t["name"] for t in r.get("tools") or []) or "-"
        print(f"{r['id']:4} {r['label']:18} {tools[:42]:42} {r['question'][:50]}")


Labels: {'OK': 15, 'PASS_REFUSE': 2}

ID   LABEL              TOOLS                                      QUESTION
--------------------------------------------------------------------------------------------------------------
q01  OK                 fulltext_search, vector_search             Do we have any newly listed known-exploited vulner
q02  OK                 fulltext_search                            What is CVE-2026-56291, what product does it affec
q03  OK                 vector_search, fulltext_search, graph_sear Are there CISA-required actions or BOD guidance ti
q04  OK                 fulltext_search                            Which recent CVEs in our sources look relevant to 
q05  OK                 vector_search                              According to NIST, how should we structure cyberse
q06  OK                 vector_search                              What does NIST CSF 2.0 say organizations should us
q07  OK                 vector_search                              

## 8. Graph-oriented relationship demos (`graph_search`)

These questions use relationship language (*tied to / issued / connected / related*) so the agent is steered toward **`graph_search`**.

| ID | Focus | Entity |
|---|---|---|
| g1 | CISA obligations / BOD | `CISA` |
| g2 | BOD 26-04 ties | `BOD 26-04` |
| g3 | SP 800-53 neighborhood | `NIST SP 800-53` |
| g4 | CSF 2.0 issued-by | `NIST Cybersecurity Framework (CSF) 2.0` |
| g5 | CMS connections | `CMS` |

**Captured run (2026-07-14):** Groq answered **g1** with `graph_search`. Daily free-tier quotas then blocked full `ask()` for g2–g5, so those cells show **raw `graph_search` evidence** (what the agent would read) plus notes. Re-run the live `demo(...)` cells after quota resets for full cited prose.


In [ ]:
# g1 — live agent answer (captured); re-run demo() when LLM quota is available
print("Q: Using our knowledge graph, what organizations, bindings-of-directives,")
print("   or related entities is CISA connected to, and what does that mean for required actions?")
print()
print("Tools:")
print("  - graph_search({'entity_name': 'CISA'})")
print("Used graph_search: True")
print()
print("""CISA is connected to the following organizations, bindings-of-directives, or related entities:
* BOD 26-04, which is a binding of directives issued by CISA, as stated in the knowledge graph (https://nvd.nist.gov/vuln/detail/CVE-2026-35273).
This means that CISA has issued BOD 26-04, which may require certain actions to be taken by organizations, as mentioned in the context of CVE-2026-35273 (https://nvd.nist.gov/vuln/detail/CVE-2026-35273).
Additionally, CISA is related to various vulnerabilities, including CVE-2026-20262 (https://nvd.nist.gov/vuln/detail/CVE-2026-20262) and CVE-2026-54420 (https://nvd.nist.gov/vuln/detail/CVE-2026-54420), which may require actions to be taken to mitigate these vulnerabilities.""")
print()
print("--- Optional live re-run (uncomment when Groq/Gemini quota is available) ---")
# demo(
#     "Using our knowledge graph, what organizations, bindings-of-directives, "
#     "or related entities is CISA connected to, and what does that mean for required actions?"
# )


Q: Using our knowledge graph, what organizations, bindings-of-directives,
   or related entities is CISA connected to, and what does that mean for required actions?

Tools:
  - graph_search({'entity_name': 'CISA'})
Used graph_search: True

CISA is connected to the following organizations, bindings-of-directives, or related entities:
* BOD 26-04, which is a binding of directives issued by CISA, as stated in the knowledge graph (https://nvd.nist.gov/vuln/detail/CVE-2026-35273).
This means that CISA has issued BOD 26-04, which may require certain actions to be taken by organizations, as mentioned in the context of CVE-2026-35273 (https://nvd.nist.gov/vuln/detail/CVE-2026-35273).
Additionally, CISA is related to various vulnerabilities, including CVE-2026-20262 (https://nvd.nist.gov/vuln/detail/CVE-2026-20262) and CVE-2026-54420 (https://nvd.nist.gov/vuln/detail/CVE-2026-54420), which may require actions to be taken to mitigate these vulnerabilities.

--- Optional live re-run (uncomment wh

### g2–g5 — raw `graph_search` evidence (LLM quota blocked full agent answers)

Re-run the next cell anytime to refresh tool output from Neo4j. Uncomment the `demo(...)` lines at the bottom once LLM quota resets.


In [ ]:
from signalpulse import retrieval as R

GRAPH_DEMOS = [
    (
        "g2",
        "What is BOD 26-04 tied to in our sources, and which entities or agencies is it related to?",
        "BOD 26-04",
        [
            "(CISA) -ISSUED_BY-> (BOD 26-04)",
            "(BOD 26-04) -APPLIES_TO-> (Cisco Unified Communications Manager)",
            "Provenance: KEV/CVE pages on BOD 26-04 patching / internet-exposure duties",
        ],
    ),
    (
        "g3",
        "What standards, controls, or policies are related to NIST SP 800-53 in our knowledge graph?",
        "NIST SP 800-53",
        [
            "(NIST) -ISSUED_BY-> (NIST SP 800-53)",
            "(NIST SP 800-53) -APPLIES_TO-> (Access Control) / ac-2 / access control policy",
            "(Executive Order 14306) -AFFECTS-> (NIST SP 800-53)",
            "Provenance: RMF page + SP 800-53 ac-2 Account Management",
        ],
    ),
    (
        "g4",
        "According to relationships in our graph, who issued NIST CSF 2.0 and what else is it connected to?",
        "NIST Cybersecurity Framework (CSF) 2.0",
        [
            "(NIST) -ISSUED_BY-> (NIST Cybersecurity Framework (CSF) 2.0)",
            "(National Institute of Standards and Technology) -ISSUED_BY-> (CSF 2.0)",
            "Provenance: https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf",
        ],
    ),
    (
        "g5",
        "What is CMS connected to in our knowledge graph, and what recent related topics show up?",
        "CMS",
        [
            "CMS PART_OF HHS; RELATED_TO OMB, VA, PRA / ACA eligibility topics",
            "Recent FR: OMB info collections, Privacy Act matching (VA assists CMS), Joint Commission",
        ],
    ),
]

for qid, question, entity, notes in GRAPH_DEMOS:
    print("=" * 72)
    print(f"{qid}: {question}")
    print(f"Expected tool: graph_search({{'entity_name': {entity!r}}})")
    print("-" * 72)
    print("Captured relationship notes:")
    for n in notes:
        print(f"  • {n}")
    print()
    print("Live graph_search() now:")
    hits = R.graph_search(entity, top_k=3)
    if not hits:
        print("  (no hits)")
    else:
        for h in hits:
            print(h.preview())
            print()

# Optional full agent re-runs (uncomment when quota allows):
# demo(GRAPH_DEMOS[0][1])
# demo(GRAPH_DEMOS[1][1])
# demo(GRAPH_DEMOS[2][1])
# demo(GRAPH_DEMOS[3][1])


## Recap & what's next

Practical employee questions are wired in `signalpulse/eval_questions.py` and scored via `run_eval.py`.

**§8 graph demos:** relationship questions that call `graph_search` (CISA/BOD, NIST SP 800-53, CSF 2.0, CMS). g1 has a full agent answer; g2–g5 show live tool evidence until LLM quota allows a re-run.

**Latest eval run:** all 15 practical questions labeled **OK** (grounded answer + citation URL); both guardrail questions **PASS_REFUSE**.

**Next — Step 9:** Streamlit chat UI over `ask()`.

> Prompt:
> *"Step 9: Create a Streamlit chat UI in `app.py` that calls the agentic RAG `ask()` function, shows answers with citations, and displays which tools were used. Keep the UI clean and beginner-friendly."*
